# Hierarchical Metadata Encoding for Wallpaper Groups

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import h5py
import json

from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
from dl_utils.utils.dataset import viz_dataloader, split_train_valid, hdf5_dataset
from dl_utils.training.build_model import resnet50_, xcit_small
from dl_utils.training.trainer import Trainer, accuracy
from dl_utils.packed_functions import benchmark_task
from dl_utils.utils.utils import list_to_dict, sort_tasks_by_size, find_last_epoch_file, viz_h5_structure


/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Inspect Input Metadata

In [10]:
input_file = Path("../../datasets/imagenet_v5_rot_10m_fix_vector.h5") # change different files here
output_file = Path("../../datasets/imagenet_v5_rot_10m_fix_vector-hierarchy_metadata.h5")

with h5py.File(input_file, "r") as h5:
    viz_h5_structure(h5)


'Group': imagenet
  'Dataset': data; Shape: (10173208, 256, 256, 3); dtype: uint8
  'Dataset': labels; Shape: (10173208,); dtype: uint8
  'Dataset': primitive_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': primitive_uc_vector_b; Shape: (10173208, 2); dtype: int32
  'Dataset': rotation_angle; Shape: (10173208,); dtype: uint8
  'Dataset': shape; Shape: (10173208,); dtype: uint8
  'Dataset': translation_start_point; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_b; Shape: (10173208, 2); dtype: int32


## 3. Build Derived Labels

In [11]:
symmetry_dict = {
    'p1': 0, 'p2': 1, 'pm': 2, 'pg': 3, 'cm': 4, 'pmm': 5,
    'pmg': 6, 'pgg': 7, 'cmm': 8, 'p4': 9, 'p4m': 10, 'p4g': 11,
    'p3': 12, 'p3m1': 13, 'p31m': 14, 'p6': 15, 'p6m': 16
}

shape_label_names = [
    'oblique',
    'hexagonal',
    'rectangular',
    'square',
    'rhombic',
    'triangle_eq',
    'triangle_r45',
    'triangle_r30',
]
shape_dict = {name: idx for idx, name in enumerate(shape_label_names)}

triangle_type_by_label = {
    'p3': 'triangle_eq',
    'p3m1': 'triangle_eq',
    'p31m': 'triangle_r30',
    'p4m': 'triangle_r45',
    'p4g': 'triangle_r45',
    'p6': 'triangle_r30',
    'p6m': 'triangle_r30',
}

symmetry_feature_labels = [
    "has_mirror",
    "has_glide",
    "has_2f",
    "has_3f",
    "has_4f",
    "has_6f",
]

symmetry_flags_map = {
    'p1':   [0, 0, 0, 0, 0, 0],
    'p2':   [0, 0, 1, 0, 0, 0],
    'pm':   [1, 0, 0, 0, 0, 0],
    'pg':   [0, 1, 0, 0, 0, 0],
    'pmm':  [1, 0, 1, 0, 0, 0],
    'pmg':  [1, 1, 1, 0, 0, 0],
    'pgg':  [0, 1, 1, 0, 0, 0],
    'cm':   [1, 0, 0, 0, 0, 0],
    'cmm':  [1, 0, 1, 0, 0, 0],
    'p4':   [0, 0, 1, 0, 1, 0],
    'p4m':  [1, 1, 1, 0, 1, 0],
    'p4g':  [1, 1, 1, 0, 1, 0],
    'p3':   [0, 0, 0, 1, 0, 0],
    'p3m1': [1, 1, 0, 1, 0, 0],
    'p31m': [0, 1, 0, 1, 0, 0],
    'p6':   [0, 0, 1, 1, 0, 1],
    'p6m':  [1, 1, 1, 1, 0, 1],
}

angle_tolerance_deg = 5.0
length_relative_tolerance = 0.05

def acute_angle_deg(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return float('nan')
    cos_theta = np.dot(vec_a, vec_b) / (norm_a * norm_b)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    theta = np.degrees(np.arccos(cos_theta))
    return theta if theta <= 90.0 else 180.0 - theta

def angle_close(value: float, target: float, tol: float = angle_tolerance_deg) -> bool:
    if np.isnan(value):
        return False
    return abs(value - target) <= tol

def lengths_close(len_a: float, len_b: float, rel_tol: float = length_relative_tolerance) -> bool:
    if len_a == 0 or len_b == 0:
        return False
    return abs(len_a - len_b) <= rel_tol * max(len_a, len_b)

def determine_shape(shape_id: int, group_name: str, vec_a: np.ndarray, vec_b: np.ndarray) -> str:
    len_a = np.linalg.norm(vec_a)
    len_b = np.linalg.norm(vec_b)
    angle = acute_angle_deg(vec_a, vec_b)

    if shape_id == 0:
        if angle_close(angle, 90.0):
            return 'square' if lengths_close(len_a, len_b) else 'rectangular'
        return 'rectangular'

    if shape_id == 1:
        if lengths_close(len_a, len_b) and angle_close(angle, 60.0):
            return 'hexagonal'
        return 'oblique'

    if shape_id == 2:
        return 'rhombic'

    if shape_id == 3:
        return triangle_type_by_label.get(group_name, 'triangle_eq')

    if lengths_close(len_a, len_b) and angle_close(angle, 60.0):
        return 'hexagonal'
    if angle_close(angle, 90.0):
        return 'square' if lengths_close(len_a, len_b) else 'rectangular'
    if lengths_close(len_a, len_b):
        return 'rhombic'
    return 'oblique'

with h5py.File(input_file, 'r') as src_h5:
    with h5py.File(output_file, 'w') as dst_h5:
        for key, value in src_h5.attrs.items():
            dst_h5.attrs[key] = value

        src_h5.copy('imagenet', dst_h5)
        dst_grp = dst_h5['imagenet']

        labels = dst_grp['labels'][:]
        original_shape_ids = dst_grp['shape'][:]
        primitive_uc_vector_a = dst_grp['primitive_uc_vector_a'][:].astype(np.float32)
        primitive_uc_vector_b = dst_grp['primitive_uc_vector_b'][:].astype(np.float32)

        index_to_group = {v: k for k, v in symmetry_dict.items()}

        derived_shape_names = [
            determine_shape(int(original_shape_ids[i]), index_to_group[int(lbl)],
                            primitive_uc_vector_a[i], primitive_uc_vector_b[i])
            for i, lbl in enumerate(labels)
        ]

        shape_labels_refined = np.array([
            shape_dict[name] for name in derived_shape_names
        ], dtype=np.uint8)

        symmetry_features = np.stack([
            symmetry_flags_map[index_to_group[int(lbl)]] for lbl in labels
        ]).astype(np.uint8)

        if 'shape_labels_refined' in dst_grp:
            del dst_grp['shape_labels_refined']
        d_shape = dst_grp.create_dataset('shape_labels_refined', data=shape_labels_refined, dtype=np.uint8)
        d_shape.attrs['description'] = (
            'Derived shape category indices using primitive vectors and wallpaper class.'
        )

        if 'symmetry_features' in dst_grp:
            del dst_grp['symmetry_features']
        d_sym = dst_grp.create_dataset('symmetry_features', data=symmetry_features, dtype=np.uint8)
        d_sym.attrs['description'] = (
            'Symmetry feature flags ordered as: ' + ', '.join(symmetry_feature_labels) + '.'
        )

        dst_h5.attrs['symmetry_dict'] = json.dumps(symmetry_dict)
        dst_h5.attrs['shape_dict'] = json.dumps(shape_dict)
        dst_h5.attrs['shape_label_names'] = json.dumps(shape_label_names)
        dst_h5.attrs['triangle_type_by_label'] = json.dumps(triangle_type_by_label)
        dst_h5.attrs['symmetry_feature_labels'] = json.dumps(symmetry_feature_labels)
        dst_h5.attrs['symmetry_flags_lookup'] = json.dumps(symmetry_flags_map)
        dst_h5.attrs['classification_parameters'] = json.dumps({
            'angle_tolerance_deg': angle_tolerance_deg,
            'length_relative_tolerance': length_relative_tolerance,
        })

        dst_h5.attrs['description'] = (
            'Wallpaper group dataset copied from the original file and enriched with derived '
            'shape labels and symmetry feature flags stored in the imagenet group.'
        )

print(f'Derived metadata stored in {output_file}')


Derived metadata stored in ../../datasets/imagenet_v5_rot_10m_fix_vector-hierarchy_metadata.h5


## 4. Normalize Geometry and Assemble Symmetry Vector

In [12]:
import h5py
import json
import numpy as np
from sklearn.preprocessing import MinMaxScaler

with h5py.File(output_file, 'a') as h5:
    grp = h5['imagenet']

    primitive_uc_vector_a = grp['primitive_uc_vector_a'][:].astype(np.float32)
    primitive_uc_vector_b = grp['primitive_uc_vector_b'][:].astype(np.float32)
    translation_start_point = grp['translation_start_point'][:].astype(np.float32)

    vectors = np.hstack([
        primitive_uc_vector_a,
        primitive_uc_vector_b,
        translation_start_point,
    ])
    scaler = MinMaxScaler()
    vectors_norm = scaler.fit_transform(vectors)

    norm_a, norm_b, norm_trans = np.split(vectors_norm, 3, axis=1)

    for name, data in {
        'normalized_primitive_uc_vector_a': norm_a,
        'normalized_primitive_uc_vector_b': norm_b,
        'normalized_translation_start_point': norm_trans,
    }.items():
        if name in grp:
            del grp[name]
        dset = grp.create_dataset(name, data=data.astype(np.float32))
        dset.attrs['description'] = (
            name.replace('_', ' ') + ' stored after MinMax scaling across the dataset.'
        )

    symmetry_features = grp['symmetry_features'][:].astype(np.float32)
    shape_labels_refined = grp['shape_labels_refined'][:].astype(np.float32)

    symmetry_vector = np.hstack([
        symmetry_features,
        shape_labels_refined.reshape(-1, 1),
        norm_a,
        norm_b,
        norm_trans,
    ]).astype(np.float32)

    if 'symmetry_vector' in grp:
        del grp['symmetry_vector']
    sym_vec = grp.create_dataset('symmetry_vector', data=symmetry_vector)
    sym_vec.attrs['description'] = (
        'Concatenated vector: symmetry features (6), shape label id (1), '
        'normalized primitive_uc_vector_a (2), primitive_uc_vector_b (2), translation_start_point (2).'
    )

    h5.attrs['geometry_vector_comment'] = (
        'symmetry_vector inside imagenet group merges symmetry hierarchy, shape id, and normalized geometry.'
    )

print('Symmetry vector regenerated inside imagenet group.')


Symmetry vector regenerated inside imagenet group.


## 5. Inspect Enriched File Structure

In [13]:
with h5py.File(output_file, "r") as h5:
    viz_h5_structure(h5)


'Group': imagenet
  'Dataset': data; Shape: (10173208, 256, 256, 3); dtype: uint8
  'Dataset': labels; Shape: (10173208,); dtype: uint8
  'Dataset': normalized_primitive_uc_vector_a; Shape: (10173208, 2); dtype: float32
  'Dataset': normalized_primitive_uc_vector_b; Shape: (10173208, 2); dtype: float32
  'Dataset': normalized_translation_start_point; Shape: (10173208, 2); dtype: float32
  'Dataset': primitive_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': primitive_uc_vector_b; Shape: (10173208, 2); dtype: int32
  'Dataset': rotation_angle; Shape: (10173208,); dtype: uint8
  'Dataset': shape; Shape: (10173208,); dtype: uint8
  'Dataset': shape_labels_refined; Shape: (10173208,); dtype: uint8
  'Dataset': symmetry_features; Shape: (10173208, 6); dtype: uint8
  'Dataset': symmetry_vector; Shape: (10173208, 13); dtype: float32
  'Dataset': translation_start_point; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'D

## 6. Inspect Stored Attributes

In [14]:
with h5py.File(output_file, "r") as h5:
            keys = [
                'symmetry_dict',
                'shape_dict',
                'shape_label_names',
                'triangle_type_by_label',
                'symmetry_feature_labels',
                'symmetry_flags_lookup',
                'classification_parameters',
            ]

            for key in keys:
                value = json.loads(h5.attrs[key])
                print(f"{key}: {value}")


symmetry_dict: {'p1': 0, 'p2': 1, 'pm': 2, 'pg': 3, 'cm': 4, 'pmm': 5, 'pmg': 6, 'pgg': 7, 'cmm': 8, 'p4': 9, 'p4m': 10, 'p4g': 11, 'p3': 12, 'p3m1': 13, 'p31m': 14, 'p6': 15, 'p6m': 16}
shape_dict: {'oblique': 0, 'hexagonal': 1, 'rectangular': 2, 'square': 3, 'rhombic': 4, 'triangle_eq': 5, 'triangle_r45': 6, 'triangle_r30': 7}
shape_label_names: ['oblique', 'hexagonal', 'rectangular', 'square', 'rhombic', 'triangle_eq', 'triangle_r45', 'triangle_r30']
triangle_type_by_label: {'p3': 'triangle_eq', 'p3m1': 'triangle_eq', 'p31m': 'triangle_r30', 'p4m': 'triangle_r45', 'p4g': 'triangle_r45', 'p6': 'triangle_r30', 'p6m': 'triangle_r30'}
symmetry_feature_labels: ['has_mirror', 'has_glide', 'has_2f', 'has_3f', 'has_4f', 'has_6f']
symmetry_flags_lookup: {'p1': [0, 0, 0, 0, 0, 0], 'p2': [0, 0, 1, 0, 0, 0], 'pm': [1, 0, 0, 0, 0, 0], 'pg': [0, 1, 0, 0, 0, 0], 'pmm': [1, 0, 1, 0, 0, 0], 'pmg': [1, 1, 1, 0, 0, 0], 'pgg': [0, 1, 1, 0, 0, 0], 'cm': [1, 0, 0, 0, 0, 0], 'cmm': [1, 0, 1, 0, 0, 0], 'p4

## 7. Sample Record

In [15]:
with h5py.File(output_file, 'r') as h5:
    grp = h5['imagenet']
    for idx in range(50, 200):

        symmetry_lookup = {int(v): k for k, v in json.loads(h5.attrs['symmetry_dict']).items()}
        shape_label_names = json.loads(h5.attrs['shape_label_names'])
        symmetry_feature_labels = json.loads(h5.attrs['symmetry_feature_labels'])

        label_id = int(grp['labels'][idx])
        group_name = symmetry_lookup[label_id]
        shape_id = int(grp['shape_labels_refined'][idx])
        symmetry_flags = grp['symmetry_features'][idx]
        symmetry_vec = grp['symmetry_vector'][idx]

        if group_name == 'p1':
                
            print(f"Sample {idx}:")
            print(f"  Wallpaper group : {group_name} ({label_id})")
            print(f"  Derived shape   : {shape_label_names[shape_id]} ({shape_id})")

            print("  Symmetry flags:")
            for name, value in zip(symmetry_feature_labels, symmetry_flags):
                print(f"    {name}: {bool(value)}")

            print("  Symmetry vector components:")
            print(f"    symmetry flags (0:6): {symmetry_vec[:6]}")
            print(f"    shape label   (6):    {symmetry_vec[6]}")
            print(f"    norm prim a   (7:9):  {symmetry_vec[7:9]}")
            print(f"    norm prim b   (9:11): {symmetry_vec[9:11]}")
            print(f"    norm trans    (11:13): {symmetry_vec[11:13]}")


Sample 51:
  Wallpaper group : p1 (0)
  Derived shape   : hexagonal (1)
  Symmetry flags:
    has_mirror: False
    has_glide: False
    has_2f: False
    has_3f: False
    has_4f: False
    has_6f: False
  Symmetry vector components:
    symmetry flags (0:6): [0. 0. 0. 0. 0. 0.]
    shape label   (6):    1.0
    norm prim a   (7:9):  [0.6282723 0.5595855]
    norm prim b   (9:11): [0.48780486 0.3393939 ]
    norm trans    (11:13): [0.397878  0.4170854]
Sample 63:
  Wallpaper group : p1 (0)
  Derived shape   : oblique (0)
  Symmetry flags:
    has_mirror: False
    has_glide: False
    has_2f: False
    has_3f: False
    has_4f: False
    has_6f: False
  Symmetry vector components:
    symmetry flags (0:6): [0. 0. 0. 0. 0. 0.]
    shape label   (6):    0.0
    norm prim a   (7:9):  [0.25130892 0.39896375]
    norm prim b   (9:11): [0.6036585 0.660606 ]
    norm trans    (11:13): [0.5039787  0.37437186]
Sample 71:
  Wallpaper group : p1 (0)
  Derived shape   : square (3)
  Symmetry flag

## 8. Tabular Preview

In [16]:
import pandas as pd

with h5py.File(output_file, 'r') as h5:
    grp = h5['imagenet']
    n = min(20, len(grp['labels']))
    symmetry_lookup = {int(v): k for k, v in json.loads(h5.attrs['symmetry_dict']).items()}
    shape_label_names = json.loads(h5.attrs['shape_label_names'])
    symmetry_feature_labels = json.loads(h5.attrs['symmetry_feature_labels'])

    df = pd.DataFrame({
        'GroupId': grp['labels'][:n],
        'Group': [symmetry_lookup[int(label)] for label in grp['labels'][:n]],
        'ShapeId': grp['shape_labels_refined'][:n],
        'ShapeLabel': [shape_label_names[int(idx)] for idx in grp['shape_labels_refined'][:n]],
    })

    symmetry_df = pd.DataFrame(
        grp['symmetry_features'][:n],
        columns=symmetry_feature_labels,
    ).astype(int)

    df = pd.concat([df, symmetry_df], axis=1)

df


,GroupId,Group,ShapeId,ShapeLabel,has_mirror,has_glide,has_2f,has_3f,has_4f,has_6f
0,15,p6,7,triangle_r30,0,0,1,1,0,1
1,6,pmg,2,rectangular,1,1,1,0,0,0
2,5,pmm,2,rectangular,1,0,1,0,0,0
3,1,p2,2,rectangular,0,0,1,0,0,0
4,4,cm,3,square,1,0,0,0,0,0
5,9,p4,3,square,0,0,1,0,1,0
6,2,pm,3,square,1,0,0,0,0,0
7,14,p31m,7,triangle_r30,0,1,0,1,0,0
8,12,p3,5,triangle_eq,0,0,0,1,0,0
9,6,pmg,2,rectangular,1,1,1,0,0,0


## 9. Verify Derived Metadata

In [17]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

with h5py.File(output_file, 'r') as h5:
    grp = h5['imagenet']
    symmetry_dict = json.loads(h5.attrs['symmetry_dict'])
    shape_label_names = json.loads(h5.attrs['shape_label_names'])
    shape_dict = json.loads(h5.attrs['shape_dict'])
    triangle_type_by_label = json.loads(h5.attrs['triangle_type_by_label'])
    symmetry_flags_lookup = json.loads(h5.attrs['symmetry_flags_lookup'])
    params = json.loads(h5.attrs['classification_parameters'])

    angle_tolerance_deg = params['angle_tolerance_deg']
    length_relative_tolerance = params['length_relative_tolerance']

    symmetry_feature_labels = json.loads(h5.attrs['symmetry_feature_labels'])

    index_to_group = {int(v): k for k, v in symmetry_dict.items()}

    labels = grp['labels'][:]
    stored_shape_labels = grp['shape_labels_refined'][:]
    stored_symmetry_features = grp['symmetry_features'][:]
    stored_symmetry_vector = grp['symmetry_vector'][:]

    primitive_uc_vector_a = grp['primitive_uc_vector_a'][:].astype(np.float32)
    primitive_uc_vector_b = grp['primitive_uc_vector_b'][:].astype(np.float32)
    translation_start_point = grp['translation_start_point'][:].astype(np.float32)
    original_shape_ids = grp['shape'][:]

    def acute_angle_deg(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
        norm_a = np.linalg.norm(vec_a)
        norm_b = np.linalg.norm(vec_b)
        if norm_a == 0 or norm_b == 0:
            return float('nan')
        cos_theta = np.dot(vec_a, vec_b) / (norm_a * norm_b)
        cos_theta = np.clip(cos_theta, -1.0, 1.0)
        theta = np.degrees(np.arccos(cos_theta))
        return theta if theta <= 90.0 else 180.0 - theta

    def angle_close(value: float, target: float, tol: float = angle_tolerance_deg) -> bool:
        if np.isnan(value):
            return False
        return abs(value - target) <= tol

    def lengths_close(len_a: float, len_b: float, rel_tol: float = length_relative_tolerance) -> bool:
        if len_a == 0 or len_b == 0:
            return False
        return abs(len_a - len_b) <= rel_tol * max(len_a, len_b)

    def determine_shape(shape_id: int, group_name: str, vec_a: np.ndarray, vec_b: np.ndarray) -> str:
        len_a = np.linalg.norm(vec_a)
        len_b = np.linalg.norm(vec_b)
        angle = acute_angle_deg(vec_a, vec_b)

        if shape_id == 0:
            if angle_close(angle, 90.0):
                return 'square' if lengths_close(len_a, len_b) else 'rectangular'
            return 'rectangular'

        if shape_id == 1:
            if lengths_close(len_a, len_b) and angle_close(angle, 60.0):
                return 'hexagonal'
            return 'oblique'

        if shape_id == 2:
            return 'rhombic'

        if shape_id == 3:
            return triangle_type_by_label.get(group_name, 'triangle_eq')

        if lengths_close(len_a, len_b) and angle_close(angle, 60.0):
            return 'hexagonal'
        if angle_close(angle, 90.0):
            return 'square' if lengths_close(len_a, len_b) else 'rectangular'
        if lengths_close(len_a, len_b):
            return 'rhombic'
        return 'oblique'

    recomputed_shape_labels = np.zeros_like(stored_shape_labels)
    recomputed_symmetry_features = np.zeros_like(stored_symmetry_features)

    for idx, label in enumerate(labels):
        group_name = index_to_group[int(label)]
        shape_name = determine_shape(
            int(original_shape_ids[idx]),
            group_name,
            primitive_uc_vector_a[idx],
            primitive_uc_vector_b[idx],
        )

        shape_idx = shape_dict[shape_name]
        recomputed_shape_labels[idx] = shape_idx
        recomputed_symmetry_features[idx] = np.array(
            symmetry_flags_lookup[group_name], dtype=np.uint8
        )

    vectors = np.hstack([
        primitive_uc_vector_a,
        primitive_uc_vector_b,
        translation_start_point,
    ])
    scaler = MinMaxScaler()
    vectors_norm = scaler.fit_transform(vectors)
    norm_a, norm_b, norm_trans = np.split(vectors_norm, 3, axis=1)

    symmetry_vector_recomputed = np.hstack([
        recomputed_symmetry_features.astype(np.float32),
        recomputed_shape_labels.reshape(-1, 1).astype(np.float32),
        norm_a,
        norm_b,
        norm_trans,
    ])

    diffs = {
        'shape_labels': np.nonzero(recomputed_shape_labels != stored_shape_labels)[0],
        'symmetry_features': np.argwhere(recomputed_symmetry_features != stored_symmetry_features),
        'symmetry_vector': np.argwhere(
            np.abs(symmetry_vector_recomputed - stored_symmetry_vector) > 1e-6
        ),
    }

    if any(len(v) > 0 for v in diffs.values()):
        print('Verification FAILED.')
        for name, idxs in diffs.items():
            if len(idxs) > 0:
                print(f"  Mismatch in {name}: first entries -> {idxs[:5]}")
    else:
        print('Verification passed: stored metadata matches recomputed values.')


Verification passed: stored metadata matches recomputed values.


### compare the file with original one to check if the copied dataset is okay, so it is safe to replace old file with new file

In [18]:
import h5py

def compare_h5_files(path_old, path_new, group="/"):
    """Compare dataset presence, shape, dtype between two HDF5 files."""
    diffs = {"missing": [], "shape": [], "dtype": []}

    with h5py.File(path_old, "r") as old_h5, h5py.File(path_new, "r") as new_h5:
        def recurse(older, newer, prefix):
            # Compare datasets in current group
            for name, obj in older.items():
                full_path = f"{prefix}/{name}".replace("//", "/")

                if name not in newer:
                    diffs["missing"].append((full_path, "missing in NEW"))
                    continue

                new_obj = newer[name]
                if isinstance(obj, h5py.Group):
                    if not isinstance(new_obj, h5py.Group):
                        diffs["dtype"].append((full_path, "expected Group, found Dataset"))
                        continue
                    recurse(obj, new_obj, full_path)
                else:
                    if not isinstance(new_obj, h5py.Dataset):
                        diffs["dtype"].append((full_path, "expected Dataset, found Group"))
                        continue

                    if obj.shape != new_obj.shape:
                        diffs["shape"].append((full_path, obj.shape, new_obj.shape))

                    if obj.dtype != new_obj.dtype:
                        diffs["dtype"].append((full_path, obj.dtype, new_obj.dtype))

            # Look for datasets/groups that exist only in new file
            for name in newer.keys():
                full_path = f"{prefix}/{name}".replace("//", "/")
                if name not in older:
                    diffs["missing"].append((full_path, "missing in OLD"))

        recurse(old_h5[group], new_h5[group], group)

    return diffs

# Example usage
old_file = "../../datasets/imagenet_v5_rot_10m_fix_vector.h5"
new_file = "../../datasets/imagenet_v5_rot_10m_fix_vector-hierarchy_metadata.h5"

diff_report = compare_h5_files(old_file, new_file, group="/")
for kind, entries in diff_report.items():
    if entries:
        print(f"\n{kind.upper()}:")
        for item in entries[:10]:
            print("  ", item)
    else:
        print(f"\n{kind.upper()}: none")



MISSING:
   ('/imagenet/normalized_primitive_uc_vector_a', 'missing in OLD')
   ('/imagenet/normalized_primitive_uc_vector_b', 'missing in OLD')
   ('/imagenet/normalized_translation_start_point', 'missing in OLD')
   ('/imagenet/shape_labels_refined', 'missing in OLD')
   ('/imagenet/symmetry_features', 'missing in OLD')
   ('/imagenet/symmetry_vector', 'missing in OLD')

SHAPE: none

DTYPE: none
